# Lab 11: Object-Oriented Design Patterns — Factory and Strategy

## Objective
Implement the Factory and Strategy design patterns in Python using Abstract Base Classes (ABCs), dataclasses, and strict type annotations for an extensible billing/payment engine.


In [1]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Dict, Type

# Strategy Pattern: Fee / Discount Calculation
class PricingStrategy(ABC):
    @abstractmethod
    def calculate_price(self, base_amount: float) -> float:
        pass

class RegularPricing(PricingStrategy):
    def calculate_price(self, base_amount: float) -> float:
        return base_amount

class StudentDiscountPricing(PricingStrategy):
    def calculate_price(self, base_amount: float) -> float:
        return base_amount * 0.80  # 20% discount

class FestivalOfferPricing(PricingStrategy):
    def calculate_price(self, base_amount: float) -> float:
        return base_amount * 0.65  # 35% discount

# Factory Pattern: Payment Processor Factory
class PaymentProcessor(ABC):
    @abstractmethod
    def process(self, amount: float) -> str:
        pass

class UPIPayment(PaymentProcessor):
    def process(self, amount: float) -> str:
        return f"Paid Rs. {amount:.2f} instantly via UPI (Zero surcharge)"

class CreditCardPayment(PaymentProcessor):
    def process(self, amount: float) -> str:
        fee = amount * 0.015
        return f"Paid Rs. {amount + fee:.2f} via Credit Card (includes 1.5% gateway fee)"

class NetBankingPayment(PaymentProcessor):
    def process(self, amount: float) -> str:
        return f"Paid Rs. {amount:.2f} via NetBanking gateway"

class PaymentProcessorFactory:
    _registry: Dict[str, Type[PaymentProcessor]] = {
        "upi": UPIPayment,
        "credit_card": CreditCardPayment,
        "netbanking": NetBankingPayment,
    }

    @classmethod
    def get_processor(cls, method: str) -> PaymentProcessor:
        processor_cls = cls._registry.get(method.lower())
        if not processor_cls:
            raise ValueError(f"Unsupported payment method: {method}")
        return processor_cls()

# Context / Client Code
@dataclass
class Order:
    order_id: str
    base_amount: float
    strategy: PricingStrategy

    def compute_final_amount(self) -> float:
        return self.strategy.calculate_price(self.base_amount)

order1 = Order("ORD101", 1000.0, StudentDiscountPricing())
final1 = order1.compute_final_amount()
processor1 = PaymentProcessorFactory.get_processor("upi")
print(f"Order: {order1.order_id} | Final: Rs. {final1:.2f} | Status: {processor1.process(final1)}")

order2 = Order("ORD102", 5000.0, FestivalOfferPricing())
final2 = order2.compute_final_amount()
processor2 = PaymentProcessorFactory.get_processor("credit_card")
print(f"Order: {order2.order_id} | Final: Rs. {final2:.2f} | Status: {processor2.process(final2)}")


Order: ORD101 | Final: Rs. 800.00 | Status: Paid Rs. 800.00 instantly via UPI (Zero surcharge)
Order: ORD102 | Final: Rs. 3250.00 | Status: Paid Rs. 3298.75 via Credit Card (includes 1.5% gateway fee)
